# 🎓 Fine-Tuning Qwen2.5-0.5B-Instruct on Presidency University OIA Knowledge
### Office of International Affairs (OIA) • Chatbot Fine-Tuning Pipeline
This notebook fine-tunes **`Qwen/Qwen2.5-0.5B-Instruct`** using **QLoRA (4-bit quantization + LoRA adapters)** with Hugging Face **PEFT** and **TRL SFTTrainer** on a free Google Colab T4 GPU.

---
### Workflow Steps:
1. **Environment Setup & GPU Check**
2. **Install Required Libraries (`peft`, `trl`, `bitsandbytes`, `transformers`, `datasets`)**
3. **Load Base Model & Tokenizer with 4-bit NormalFloat (NF4) Quantization**
4. **Configure LoRA Hyperparameters (Rank 16, Alpha 32)**
5. **Load and Format Presidency University OIA Instruction Dataset**
6. **Train with SFTTrainer & Monitor Convergence**
7. **Save Adapter Weights & Run Interactive Test Inquiries**


In [ ]:
# Check available GPU
!nvidia-smi


In [ ]:
# Install Hugging Face & Fine-Tuning packages
!pip install -q -U torch transformers peft trl datasets accelerate bitsandbytes


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Loading base model: {MODEL_ID} ...")

# 4-bit quantization configuration for low VRAM consumption (~2.5 GB on T4 GPU)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

print("Base model loaded with 4-bit NF4 quantization!")


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# Prepare model for 4-bit training
model = prepare_model_for_kbit_training(model)

# LoRA adapter parameters targeting attention and projection layers
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()


In [ ]:
# Upload or load data/finetune_dataset.jsonl
import json
from datasets import Dataset

# If running on Colab, upload finetune_dataset.jsonl or use the raw records below
dataset_path = "finetune_dataset.jsonl"

try:
    with open(dataset_path, "r", encoding="utf-8") as f:
        records = [json.loads(line) for line in f if line.strip()]
except FileNotFoundError:
    print("Uploading / Fallback inline dataset sample...")
    records = [
        {
            "prompt": "You are the official AI Counselor at Presidency University. How many days does an international student have to register with FRRO?",
            "completion": "Under Government of India regulations, all foreign national students holding a visa for more than 180 days must register with the FRRO within 14 days of landing in India via indianfrro.gov.in. Presidency University OIA issues the required Bonafide Certificate and Form C."
        },
        {
            "prompt": "What visa is mandatory for international students joining Presidency University?",
            "completion": "International students must enter India strictly on a valid 'Student Visa' (S-Visa) that explicitly mentions 'Presidency University'. Tourist or business visas cannot be converted."
        },
        {
            "prompt": "What scholarships are offered to foreign students at Presidency University?",
            "completion": "Presidency University supports Indian Council for Cultural Relations (ICCR) scholarships, Study In India (SII) merit waivers up to 50%-100%, and University International Merit Scholarships of 15% to 30% for high academic achievers."
        }
    ]

# Format as chat text for SFTTrainer
def format_chat_prompt(item):
    text = f"<|im_start|>user\n{item['prompt']}<|im_end|>\n<|im_start|>assistant\n{item['completion']}<|im_end|>"
    return {"text": text}

raw_dataset = Dataset.from_list(records)
train_dataset = raw_dataset.map(format_chat_prompt)
print(f"Loaded {len(train_dataset)} training examples!")
print("Sample item:\n", train_dataset[0]["text"][:250])


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

output_dir = "./fine_tuned_model"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    fp16=True,
    logging_steps=5,
    save_strategy="epoch",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    report_to="none"
)

trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer
)

print("Starting LoRA fine-tuning...")
trainer.train()

# Save fine-tuned LoRA adapters
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved successfully to {output_dir}!")


In [ ]:
# Test inference with fine-tuned model
from transformers import pipeline

test_prompt = "What is the mandatory FRRO registration timeframe for an international student at Presidency University?"

messages = [
    {"role": "system", "content": "You are the official AI Counselor for the Office of International Affairs at Presidency University."},
    {"role": "user", "content": test_prompt}
]

prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== Student Inquiry ===")
print(test_prompt)
print("\n=== Fine-Tuned Model Response ===")
print(response.strip())
